# Fingerprint improvement sweep — round 6

Round 5 was the first round in which **no variant won** (full record in `DIARY.md`). It settled
three things while doing so:

1. **The activity axis is exhausted, and it turns over.** Round 4 found EER tracking the
   dead-unit fraction at r = +0.96. Round 5 pushed activity from five directions: `sust_2` and
   `scale_2` drove `dead` to the loop's lowest values (1.1%, 0.7%) and got **worse**. The
   relationship is an inverted U and the champion, at ~2% dead, sits at its bottom.
2. **Round 4's smaller winners do not add.** `inh_weak` and `stdp_2x` beat the *old* champion by
   2.3x and 2.1x the noise floor; re-scored on top of `enc_sust` they collapse to 1.1x and 0.1x
   ties. They were alternative routes to the same activity increase.
3. **The encoder win was about the sustained channel's absolute level, not the ratio.**
   `onset_1` gives a *more* sustained-dominant balance than the champion and is the worst row of
   the round (7.9x floor). `sust_gain=1.0` is the optimum. The encoder is closed.

**The round's real find was free:** nuisance projection started working. `proj-20` hurt every
variant in round 2; it now helps **8 of 9** by a consistent −0.019 (≈5x floor), and
`vth_035 + proj-20 = 0.3125` is the best generalisation number of the loop. Likely cause: at
15% dead the within-speaker subspace was partly *structural*, so projecting it out removed
signal; at 2% dead it is genuinely nuisance. This needs no session labels at test time.

**The stopping-rule counter is at 1 of 2.** Another round with no >2x-floor winner ends the loop.

Round 6 therefore leaves the activity family and screens the four never varied in five rounds.

| variant | change | family / question |
|---|---|---|
| `champ` / `champ_rep` | — | reference + noise floor + reproducibility |
| `vth_nexc` | `vth_rest` 0.35 + `norm_limit_exc` 1.5 | the two round-5 sub-threshold trends, combined |
| `beta_lo` | `beta` 1.5 | **input-layer adaptation strength** — novelty vs steady tone |
| `tau_a_120` | `tau_a_ms` 120 | how long that adaptation lasts |
| `vthj_2` | `vth_jump` 2.0 | **hidden spike-frequency adaptation** — a *competition* mechanism |
| `tau_vth_200` | `tau_vth_ms` 200 | how long that competition persists |
| `triplet_off` | `a3pre`/`a3post` 0 | is the triplet STDP term earning its place? |
| `p_exc_1` | `p_exc` 1 | **topographic profile** — sharpen the receptive field without cutting fan-in |

`vth_nexc` is a deliberate exception to one-factor-at-a-time, flagged so it is not mistaken for
drift: both knobs were sub-threshold on EER but agreed across EER, R@1 and mAP.

**Nothing is cached and nothing needs attaching.** The champion is re-run rather than reused:
the clip set is fixed by `SEED`, so its number stays comparable to round 5's 0.3330, and since
the hidden neurons carry unseeded noise, the gap between the pair *is* the run-to-run noise
floor. It has come back at 0.0031-0.0035 four rounds running.

**`PC1-5` is not comparable across variants** (the weight tensors change dimension with
`r_exc`). Use **`hid PC5`**, which is always 128-d.

Runs locally (`datasets/vox1`) or on Kaggle; edit `INPUT_ROOT` / `DEV_NN_STYLE` in CONFIG.
The `%%writefile` cells recreate the extractor as real modules so `spawn` workers can import
them (a notebook has no importable `__main__`).


In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
# On Kaggle, uncomment the installs. Locally the repo venv already has these.
!pip -q install gammatone || pip -q install git+https://github.com/detly/gammatone.git
!pip -q install brian2
import shutil
import brian2, gammatone, librosa, soundfile
assert shutil.which("ffmpeg"), "ffmpeg not on PATH — needed to decode .m4a"
print("brian2", brian2.__version__, "| ffmpeg", shutil.which("ffmpeg"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG — the only cell you normally edit
# ══════════════════════════════════════════════════════════════════════════════
import os

# Corpus root. Two supported layouts:
#   DEV_NN_STYLE=True :  {ROOT}/dev_NN/{person}/{session}/{utt}.m4a   (VoxCeleb1 as shipped)
#   DEV_NN_STYLE=False:  {ROOT}/{person}/{session}/{utt}.m4a          (a single shard)
def _find_corpus():
    """Walk up from the working directory looking for datasets/vox1, so the notebook
    works whether it is run from its own folder or from the repo root. On Kaggle this
    finds nothing and falls back to the dataset mount — edit it there."""
    d = os.path.abspath(os.getcwd())
    for _ in range(5):
        c = os.path.join(d, "datasets", "vox1")
        if os.path.isdir(c):
            return c
        d = os.path.dirname(d)
    return "/kaggle/input/datasets/qphulong/vox1-voices"

INPUT_ROOT   = _find_corpus()           # or hard-code a shard path
DEV_NN_STYLE = True
AUDIO_EXTS   = (".m4a", ".wav")

OUT_DIR = "sweep_out"          # one npz per variant lands here

# Sample size. Cost scales linearly with N_WAVS = N_SPEAKERS * SESSIONS * UTTS.
# UTTS_PER_SESSION > 1 is what makes the same-session convergence measurement possible —
# the main corpus protocol stores only one utterance per session, so this sweep is the
# first chance to test that claim. Do not set it to 1.
N_SPEAKERS        = 60
SESSIONS_PER_SPK  = 4
UTTS_PER_SESSION  = 2
SEED              = 1234

WORKERS = os.cpu_count() or 2

# ── The variants being compared (see DIARY.md, round 6) ───────────────────────
# Every variant runs over the SAME clips, so all comparisons are paired.
# One factor at a time: each variant is the champion plus exactly ONE change, so an
# effect is attributable. Every knob in _fp_sweep_core.DEFAULTS is overridable, and
# unknown keys raise rather than being silently ignored — a typo costs a traceback
# instead of a whole run.
#
# Round 5 result: NO variant won. The activity axis that explained round 4 turns over —
# `sust_2` and `scale_2` drove dead units to the loop's lowest values (1.1%, 0.7%) and
# got WORSE, so the champion at ~2% dead sits at the bottom of an inverted U. Round 4's
# two smaller winners also collapsed to ties on top of the new champion: they were
# alternative routes to the same activity increase, not independent mechanisms.
# The stopping-rule counter is at 1 of 2.
#
# So round 6 abandons the activity family and screens the four never varied in five
# rounds: input-layer adaptation, hidden spike-frequency adaptation, the triplet STDP
# term, and the topographic profile shape.
CHAMP = dict(n_epochs=2, r_exc=5, vth_rest=0.45, clip_ms=8000, sust_gain=1.0)

# `o` overrides CHAMP, so a variant may re-set a key the champion already fixes.
_V = lambda name, **o: dict(name=name, **{**CHAMP, **o})

VARIANTS = [
    _V("champ"),                              # reference (round-5 pair mean 0.3330)
    _V("champ_rep"),                          # deliberate replicate -> noise floor
    # the one deliberate combination: both were sub-threshold on EER in round 5 but
    # agreed across EER, R@1 and mAP, which is the case where combining earns a slot
    _V("vth_nexc",    vth_rest=0.35, norm_limit_exc=1.5),
    # -- input-layer adaptive LIF: never varied in five rounds --
    _V("beta_lo",     beta=1.5),              # 3.5 : adaptation strength (novelty vs steady)
    _V("tau_a_120",   tau_a_ms=120.0),        # 40  : how long that adaptation lasts
    # -- hidden spike-frequency adaptation: a COMPETITION mechanism, not a gain knob.
    #    It silences the most active units selectively, unlike everything round 5 tested.
    _V("vthj_2",      vth_jump=2.0),          # 1.0 : strength of the winner-take-all
    _V("tau_vth_200", tau_vth_ms=200.0),      # 60  : how long the competition persists
    # -- is the triplet STDP term earning its place? --
    _V("triplet_off", a3pre=0.0, a3post=0.0), # 0.004 / -0.002
    # -- topographic profile max(0, 1-(d/r)^p): p=3 near-boxcar, p=1 a linear cone.
    #    Sharpens the receptive field WITHOUT cutting fan-in — the defect-1 fix that
    #    r_exc could not deliver without killing units.
    _V("p_exc_1",     p_exc=1),               # 3
]

# Pairs with identical config, used to separate simulation noise from real structure.
REPLICATES = [("champ", "champ_rep")]

os.makedirs(OUT_DIR, exist_ok=True)

# ── Seed the cache from an attached data source ───────────────────────────────
# A Kaggle commit starts from an EMPTY working dir, so `sweep_out/` does not survive
# between runs. To reuse a previous round:
#     Add Data -> Your Work -> the earlier notebook version
# It mounts under /kaggle/input/<slug>/, keeping the sweep_out/ folder, and this
# copies any fp_*.npz it finds into OUT_DIR. Search is depth-bounded and prunes any
# directory holding audio — a naive recursive walk would crawl the whole vox1 corpus.
import shutil

def _seed_cache(out_dir, root="/kaggle/input", max_depth=4):
    found = []
    if not os.path.isdir(root):
        return found
    base = root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root):
        for f in filenames:
            if f.startswith("fp_") and f.endswith(".npz"):
                dst = os.path.join(out_dir, f)
                if not os.path.exists(dst):
                    shutil.copy(os.path.join(dirpath, f), dst)
                found.append(f[3:-4])
        if (dirpath.rstrip("/").count("/") - base >= max_depth
                or any(x.lower().endswith(AUDIO_EXTS) for x in filenames[:5])):
            dirnames[:] = []          # stop descending: too deep, or this is the corpus
    return sorted(set(found))

# Off by default: the protocol re-runs the champion on purpose (see VARIANTS), and a
# stale attached data source would silently satisfy it from cache and destroy that
# measurement. Set True only to resume an interrupted round.
SEED_CACHE_FROM_INPUT = False

_seeded = _seed_cache(OUT_DIR) if SEED_CACHE_FROM_INPUT else []
if _seeded:
    print(f"seeded   : {_seeded}")
elif SEED_CACHE_FROM_INPUT:
    _mounts = sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else []
    print("seeded   : nothing found to resume from")
    print(f"           (mounts seen: {_mounts or 'none'}; to reuse a round, "
          f"Add Data -> Your Work -> that notebook version)")
else:
    print("seeded   : disabled — every variant runs fresh (SEED_CACHE_FROM_INPUT=False)")

_n_wavs = N_SPEAKERS * SESSIONS_PER_SPK * UTTS_PER_SESSION
# Cost model calibrated on round 1 (4 workers): 0.41s fixed per wav (decode + encode)
# plus 0.116s per 2s-clip exposure. One "unit" = n_epochs * clip_ms/2000.
_units = lambda v: v["n_epochs"] * v.get("clip_ms", 2000) / 2000
_todo = [v for v in VARIANTS
         if not os.path.exists(os.path.join(OUT_DIR, f"fp_{v['name']}.npz"))]
_eta = sum(_n_wavs * (0.41 + 0.116 * _units(v)) for v in _todo) * 4 / max(WORKERS, 1)
print(f"corpus   : {INPUT_ROOT}  (dev_NN layout = {DEV_NN_STYLE})")
print(f"sample   : {N_SPEAKERS} speakers x {SESSIONS_PER_SPK} sessions x "
      f"{UTTS_PER_SESSION} utts = {_n_wavs} wavs")
print(f"variants : {[v['name'] for v in VARIANTS]}")
print(f"workers  : {WORKERS}")
print(f"rough ETA: {_eta/60:.0f} min for {len(_todo)} variant(s) still to run, "
      f"plus ~1 min Cython compile per distinct vth_rest")
print(f"output   : {OUT_DIR}")


### Write the extractor modules to the working dir (imported by spawn workers)

In [ ]:
%%writefile audio_utils.py
import shutil
import subprocess

import numpy as np
import librosa
import soundfile as sf
from gammatone.filters import centre_freqs, make_erb_filters, erb_filterbank


def _ffprobe_sample_rate(path):
    """Native sample rate of `path` via ffprobe, or None if it can't be determined."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None:
        return None
    try:
        out = subprocess.run(
            [ffprobe, "-v", "error", "-select_streams", "a:0",
             "-show_entries", "stream=sample_rate",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            check=True, capture_output=True, text=True,
        ).stdout.strip().splitlines()
        return int(out[0]) if out else None
    except (subprocess.CalledProcessError, ValueError):
        return None


def _ffmpeg_decode(path, sr):
    """Decode any ffmpeg-readable container (m4a/aac/mp3/...) to a mono float32
    waveform. If `sr` is None the native rate is probed and preserved; otherwise
    ffmpeg's resampler outputs directly at `sr`."""
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            f"Cannot decode {path!r}: libsndfile failed and ffmpeg is not on PATH. "
            "Install ffmpeg to read m4a/aac audio."
        )
    target_sr = sr if sr is not None else (_ffprobe_sample_rate(path) or 16000)
    proc = subprocess.run(
        [ffmpeg, "-nostdin", "-loglevel", "error", "-i", path,
         "-ac", "1", "-ar", str(target_sr),
         "-f", "f32le", "-acodec", "pcm_f32le", "-"],
        capture_output=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"ffmpeg failed to decode {path!r}: "
            f"{proc.stderr.decode('utf-8', 'ignore').strip()}"
        )
    y = np.frombuffer(proc.stdout, dtype="<f4").astype(np.float32)
    return y, target_sr


def load_audio(path, sr=16000):
    """Load `path` to a mono float32 waveform at `sr` Hz (native rate if `sr` is None).

    Format-robust replacement for ``librosa.load``: wav/flac/ogg are decoded by
    libsndfile (soundfile); m4a/aac and any container libsndfile can't open are
    decoded via ffmpeg. This avoids librosa's audioread m4a fallback, which is
    deprecated and slated for removal in librosa 1.0 (and emits a warning per file).
    """
    try:
        y, sr_native = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(path, sr)
    if y.ndim > 1:                       # multi-channel -> mono (match librosa default)
        y = y.mean(axis=1)
    if sr is not None and sr_native != sr:
        y = librosa.resample(y, orig_sr=sr_native, target_sr=sr)
    return np.ascontiguousarray(y, dtype=np.float32), (sr if sr is not None else sr_native)


def load_mel_spectrogram(
    wav_path: str,
    n_mels: int = 256,
    fmax: int | None = 8000,
    target_frames_per_second: int = 1000,
    normalize: bool = True,
):
    audio, sr = load_audio(wav_path, sr=None)

    hop_length = int(sr / target_frames_per_second)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        fmax=fmax,
        hop_length=hop_length
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if normalize:
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    return mel_db, sr

def auditory_frontend(
    audio_path,
    sr=16000,
    num_filters=100,
    f_min=50,
    alpha=1.0,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
    eps=1e-6,
):
    """
    Encode an audio waveform into auditory-inspired spike features

    This function implements a biologically inspired auditory pipeline:
    waveform → gammatone filterbank → upstream percentile normalization →
    inner hair cell compression → onset detection → phase signal.

    Normalization is applied **once, upstream** to the (signed) filterbank output,
    before any nonlinearity. Because `log1p(alpha * x)` is not scale-invariant, the
    signal must be brought to a known scale *before* the log so compression is
    consistent across utterances. E, dE and phase are then all derived from the same
    normalized signal — their relative balance is therefore set only by the downstream
    gains, not by independent per-feature normalizations.

    Parameters
    ----------
    audio_path : str
        Path to the input audio file.

    sr : int, default=16000
        Target sampling rate for loading audio.

    num_filters : int, default=100
        Number of ERB-spaced gammatone filters (frequency channels).

    f_min : float, default=50
        Minimum center frequency (Hz) of the filterbank.

    alpha : float, default=1.0
        Compression strength for inner hair cell log compression:
        E = log1p(alpha * |signal_norm|).

    norm_percentile : float, default=99.0
        Percentile of |filterbank output| used as the normalization scale. Robust to
        the loudest transients (top 1% at 99) compared to a plain max.

    clip_val : float, default=1.0
        After dividing by the percentile scale, the normalized signal is clipped to
        [-clip_val, clip_val]. This bounds the input to the log and saturates the
        loudest excursions.

    per_channel : bool, default=False
        If False (default), one global percentile scalar is computed over the whole
        (n_channels, T) magnitude array — this preserves cross-channel relative energy
        (formant/timbre structure useful for speaker discrimination). If True, the
        percentile is computed per channel, equalizing quiet and loud bands.

    eps : float, default=1e-6
        Small constant to avoid division by zero.

    Returns
    -------
    dict
        Dictionary containing encoded auditory representations:

        - "E" : np.ndarray (n_channels, T)
            Log-compressed cochlear energy (IHC output), full-wave rectified.

        - "dE" : np.ndarray (n_channels, T)
            Onset detection signal (half-wave rectified temporal derivative of E).

        - "phase" : np.ndarray (n_channels, T)
            Negative half-wave of the normalized filterbank output. Complementary in
            polarity to E's full-wave energy, so it is not redundant with E.

        - "cf" : np.ndarray (n_channels,)
            Center frequencies of filterbank channels (low → high)

        - "sr" : int
            Sampling rate of processed audio

    Notes
    -----
    Processing pipeline:

    1. Audio loading
    2. ERB-spaced gammatone filterbank
    3. Upstream percentile normalization + clip (on the signed signal)
    4. Inner hair cell log compression (full-wave): E = log1p(alpha * |sig_norm|)
    5. Onset detection via positive temporal derivative of E
    6. Phase signal: negative half-wave of sig_norm

    All channel outputs are ordered from **low → high frequency**.
    """

    # ==============================
    # 1. Load audio
    # ==============================
    signal, sr = load_audio(audio_path, sr=sr)

    # ==============================
    # 2. Gammatone filterbank
    # ==============================
    cf = centre_freqs(sr, num_filters, f_min)
    erb_filters = make_erb_filters(sr, cf)

    filtered_signals = erb_filterbank(signal, erb_filters)

    # reorder HIGH→LOW → LOW→HIGH
    cf = cf[::-1]
    filtered_signals = filtered_signals[::-1]

    signals = filtered_signals
    n_channels, T = signals.shape

    # ==============================
    # 3. Upstream percentile normalization (on the signed signal, before any
    #    nonlinearity). One scale derived from |signals|, then clip. This keeps
    #    the log compression in a consistent regime across utterances and puts
    #    E / dE / phase on a single shared reference frame.
    # ==============================
    if per_channel:
        scale = np.percentile(np.abs(signals), norm_percentile, axis=1, keepdims=True)
    else:
        scale = np.percentile(np.abs(signals), norm_percentile)
    sig_n = np.clip(signals / (scale + eps), -clip_val, clip_val)

    # ==============================
    # 4. Inner Hair Cell Compression (full-wave)
    # ==============================
    E = np.log1p(alpha * np.abs(sig_n))

    # ==============================
    # 5. Onset detection (positive temporal derivative of E)
    # ==============================
    dE = np.diff(E, axis=1, prepend=E[:, :1])
    dE[dE < 0] = 0

    # ==============================
    # 6. Phase signal: negative half-wave of the normalized signal.
    #    Complementary in polarity to E's full-wave energy → not redundant with E.
    # ==============================
    phase_signal = np.maximum(-sig_n, 0)

    return {
        "E": E,
        "dE": dE,
        "phase": phase_signal,
        "cf": cf,
        "sr": sr,
    }


In [ ]:
%%writefile spike_encoding.py
import numpy as np
from audio_utils import auditory_frontend

def compute_spike_input_current(
    audio_path,
    sustained_per_band=5,
    onset_per_band=2,
    phase_per_band=2,
    scale=1,
    sust_gain=1.0,
    onset_gain=2.0,
    phase_gain=1.0,
    sust_spread_min=0.6,
    sust_spread_max=1.4,
    audio_sample_rate=16000,
    simulation_sample_rate=1000,
    num_filters=100,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
):
    """
    Convert an audio file into a downsampled input current matrix for a spiking neural network

    This function takes auditory features produced by `auditory_frontend()` and expands
    them into multiple neuron types per cochlear frequency band. Each neuron type
    represents different auditory response characteristics inspired by biological
    auditory nerve fibers.

    Pipeline
    --------
    1. Audio → auditory feature extraction via `auditory_frontend()`
    2. Obtain three feature maps:
        - E     : sustained energy (IHC compressed output)
        - dE    : onset energy (positive temporal derivative)
        - phase : rectified gammatone signal
    3. Generate multiple neurons per frequency band:
        - sustained neurons (energy response, spread across gain multipliers)
        - onset neurons (transient response)
        - phase neurons (phase locking)
    4. Apply gain scaling and small Gaussian noise.
    5. Downsample from `audio_sample_rate` to `simulation_sample_rate` by
       block-averaging across the decimation factor.
    6. Return a time-varying current matrix suitable for driving LIF neurons.

    Parameters
    ----------
    audio_path : str
        Path to the input audio (.wav) file.

    sustained_per_band : int, default=5
        Number of neurons per frequency band that encode sustained energy (E).

    onset_per_band : int, default=2
        Number of neurons per band that encode onset activity (dE).

    phase_per_band : int, default=2
        Number of neurons per band that encode phase-locking signals.

    scale : float, default=1
        Global gain multiplier applied to all input currents.

    sust_gain : float, default=1.3
        Gain factor for sustained-energy neurons.

    onset_gain : float, default=2.0
        Gain factor for onset neurons.

    phase_gain : float, default=1.0
        Gain factor for phase-locking neurons.

    sust_spread_min : float, default=0.6
        Minimum multiplicative factor applied across sustained neurons within a band.

    sust_spread_max : float, default=1.4
        Maximum multiplicative factor applied across sustained neurons within a band.

    audio_sample_rate : int, default=16000
        Sampling rate (Hz) of the raw audio and the auditory feature maps produced
        by `auditory_frontend()`.

    simulation_sample_rate : int, default=1000
        Target sampling rate (Hz) for the Brian2 simulation (i.e. 1 / defaultclock.dt).
        The current matrix is downsampled from `audio_sample_rate` to this rate by
        block-averaging. Must evenly divide `audio_sample_rate`.

    norm_percentile : float, default=99.0
        Percentile used by `auditory_frontend` to normalize the filterbank output
        before the log nonlinearity.

    clip_val : float, default=1.0
        Clip bound applied to the normalized filterbank signal in `auditory_frontend`.

    per_channel : bool, default=False
        If True, `auditory_frontend` normalizes per channel instead of globally.

    Returns
    -------
    I_sim : np.ndarray, shape (N_in, T_sim)
        Simulation-ready input current matrix, where
        T_sim = T // decimation_factor.

    T_sim : int
        Number of time steps after downsampling, corresponding to the
        total simulation duration in Brian2 timesteps.
    """

    feats = auditory_frontend(
        audio_path,
        num_filters=num_filters,
        norm_percentile=norm_percentile,
        clip_val=clip_val,
        per_channel=per_channel,
    )

    E = feats["E"]
    dE = feats["dE"]
    phase = feats["phase"]

    n_channels, T = E.shape

    g_sust = sust_gain
    g_onset = onset_gain
    g_phase = phase_gain

    neurons_per_band = sustained_per_band + onset_per_band + phase_per_band
    N_in = n_channels * neurons_per_band

    I = np.zeros((N_in, T), dtype=np.float32)

    idx = 0
    for ch in range(n_channels):

        spread = np.linspace(sust_spread_min, sust_spread_max, sustained_per_band)
        for mult in spread:
            I[idx] = g_sust * mult * scale * E[ch]
            idx += 1

        for _ in range(onset_per_band):
            I[idx] = g_onset * scale * dE[ch]
            idx += 1

        for _ in range(phase_per_band):
            I[idx] = g_phase * scale * phase[ch]
            idx += 1

    I += 0.01 * np.random.randn(*I.shape).astype(np.float32)
    assert audio_sample_rate % simulation_sample_rate == 0, (
        f"audio_sample_rate ({audio_sample_rate}) must be divisible by "
        f"simulation_sample_rate ({simulation_sample_rate})"
    )
    decimation_factor = audio_sample_rate // simulation_sample_rate
    # Trim to nearest multiple so reshape never fails
    T_trim = (T // decimation_factor) * decimation_factor
    I_sim  = I[:, :T_trim].reshape(N_in, -1, decimation_factor).mean(axis=2)
    T_sim  = I_sim.shape[1]

    return I_sim, T_sim


In [ ]:
%%writefile _fp_sweep_core.py
"""
_fp_sweep_core.py
=================
Variant-parameterised fingerprint extractor for the improvement sweep.

Same network as `ecapa_film_snn/prepare_fingerprints.ipynb`'s `_fingerprint_core.py`
(which mirrors `tonotopic_plasticity_bound/train.py`). Rounds 1-3 exposed only
n_epochs / r_exc / vth_rest / clip_ms; from round 4 EVERY hyperparameter in DEFAULTS
is overridable per variant, so the sweep can screen the whole space rather than the
four knobs that happened to be wired up.

Constants now reach Brian2 through per-object `namespace=` dicts instead of being
baked into the equation strings as f-string literals. Two consequences:
  * any of them can vary per variant without touching the model text, and
  * the Cython code compiles ONCE for the whole sweep rather than once per distinct
    vth_rest, which is why warm-up is now a one-off cost.

At DEFAULTS this is numerically identical to the round-3 champion.

Import BEFORE numpy in driver code so the BLAS pinning below takes effect.
"""

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import sys
from types import SimpleNamespace

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
from spike_encoding import compute_spike_input_current

from brian2 import (
    NeuronGroup, Synapses, SpikeMonitor, TimedArray, Network, network_operation,
    defaultclock, prefs, BrianLogger, ms, second,
)

prefs.codegen.target = 'cython'
prefs.codegen.runtime.cython.multiprocess_safe = True
BrianLogger.suppress_name('method_choice')
BrianLogger.suppress_name('unused_brian_object')
prefs.logging.console_log_level = 'ERROR'

# ── Fixed structure (changing these changes tensor shapes; not swept) ─────────
N_IN = 128
N_H  = 128
N_CHANNELS    = 64
N_PER_CHANNEL = N_IN // N_CHANNELS
DT_SIM = 1 * ms

# ── Everything sweepable. Values are the round-3 champion. ───────────────────
DEFAULTS = dict(
    # -- exposed in rounds 1-3 --
    n_epochs=2, r_exc=5, vth_rest=0.45, clip_ms=8000,
    # -- input (adaptive LIF) layer --
    tau_m_ms=40.0, tau_a_ms=40.0, beta=3.5, tau_current_ms=1.0, v_th_in=1.0,
    # -- hidden layer --
    tau_h_ms=150.0, tau_vth_ms=60.0, vth_jump=1.0, tau_r_ms=10.0, sigma_noise=0.03,
    # -- excitatory STDP (pair term + triplet term) --
    apre=0.008, apost=-0.0096, taupre_ms=20.0, taupost_ms=20.0,
    a3pre=0.004, a3post=-0.002, tau_x_ms=100.0, tau_y_ms=125.0, wmax=1.0,
    # -- inhibitory STDP --
    apre_inh=0.004, apost_inh=-0.0048, w_inh_center=1.0,
    # -- weight normalisation (the L1 column budget is the binding constraint) --
    norm_limit_exc=1.0455, norm_limit_inh=0.40, norm_dt_ms=25.0,
    # -- topology --
    p_exc=3,
    # -- auditory front-end --
    enc_scale=1.0, sust_gain=0.3, onset_gain=3.0,
)

WMIN = 0.0
W_INH_MIN = 0.0


def encoder_kwargs(P):
    return dict(scale=P.enc_scale, num_filters=N_CHANNELS,
                sustained_per_band=1, onset_per_band=1, phase_per_band=0,
                sust_gain=P.sust_gain, onset_gain=P.onset_gain,
                sust_spread_min=1, sust_spread_max=1)


# ═══════════════════════════════════════════════════════════════════════════════
# Per-variant precompute
# ═══════════════════════════════════════════════════════════════════════════════

def make_params(name="variant", **over):
    """All variant-dependent matrices. Unknown keys raise rather than being ignored
    silently — a typo in a sweep config would otherwise cost a whole run."""
    bad = set(over) - set(DEFAULTS)
    if bad:
        raise KeyError(f"unknown hyperparameter(s) {sorted(bad)}; "
                       f"valid: {sorted(DEFAULTS)}")
    cfg = {**DEFAULTS, **over}
    P = SimpleNamespace(name=name, **cfg)

    r_exc = P.r_exc
    ch_i = (np.arange(N_IN) // N_PER_CHANNEL).reshape(-1, 1)
    ch_j = (np.arange(N_H) // N_PER_CHANNEL).reshape(1, -1)
    dist = np.abs(ch_i - ch_j)
    dist = np.minimum(dist, N_CHANNELS - dist)

    topo = np.maximum(0.0, 1.0 - (dist / r_exc) ** P.p_exc)
    mask_ih = dist <= r_exc
    src_ih, tgt_ih = np.where(mask_ih)

    # Inhibitory Jaccard connectivity, window tied to the same radius.
    ch_h = np.arange(N_H) // N_PER_CHANNEL
    d_hh = np.abs(ch_h.reshape(-1, 1) - ch_h.reshape(1, -1))
    d_hh = np.minimum(d_hh, N_CHANNELS - d_hh)
    window = 2 * r_exc + 1
    overlap = np.maximum(0, window - d_hh)
    jac = np.where(overlap > 0, overlap / (window + d_hh), 0.0)
    mask_hh = (overlap > 0) & (~np.eye(N_H, dtype=bool))
    src_hh, tgt_hh = np.where(mask_hh)

    w_ih_init = np.zeros((N_IN, N_H))
    w_ih_init[src_ih, tgt_ih] = (P.wmax * topo)[src_ih, tgt_ih]
    for j in range(N_H):
        rows = src_ih[tgt_ih == j]
        s = w_ih_init[rows, j].sum()
        if s > 0:
            w_ih_init[rows, j] *= P.norm_limit_exc / s

    rng = np.random.RandomState(42)
    w_hh_init = np.zeros((N_H, N_H))
    w_hh_init[src_hh, tgt_hh] = rng.uniform(0.01, 0.02, size=src_hh.shape[0])

    offsets = np.arange(-r_exc, r_exc + 1)
    ch_row = np.arange(N_H) // N_PER_CHANNEL
    off_ch = (ch_row[None, :] + offsets[:, None]) % N_CHANNELS
    idx = np.stack([off_ch * N_PER_CHANNEL + t for t in range(N_PER_CHANNEL)])
    shape = (N_PER_CHANNEL, len(offsets), N_H)

    P.src_ih, P.tgt_ih, P.src_hh, P.tgt_hh = src_ih, tgt_ih, src_hh, tgt_hh
    P.wmax_m   = P.wmax * topo
    P.apre_m   = P.apre * topo
    P.apost_m  = P.apost * topo
    P.a3pre_m  = P.a3pre * topo
    P.a3post_m = P.a3post * topo
    P.wmax_inh   = P.w_inh_center * jac
    P.apre_inh_m  = P.apre_inh * jac
    P.apost_inh_m = P.apost_inh * jac
    P.w_ih_init, P.w_hh_init = w_ih_init, w_hh_init
    P.idx = idx
    P.j_row = np.broadcast_to(np.arange(N_H), shape).copy()
    P.shape = shape
    return P


# ═══════════════════════════════════════════════════════════════════════════════
# Network
# ═══════════════════════════════════════════════════════════════════════════════

def build_network(P):
    defaultclock.dt = DT_SIM

    ns_in = dict(tau_m=P.tau_m_ms * ms, tau_a=P.tau_a_ms * ms,
                 tau_current=P.tau_current_ms * ms, v_th_in=P.v_th_in, beta=P.beta)
    eqs_in = """
    dv/dt = (-v - a) / tau_m + I_timed(t, i) / tau_current : 1
    da/dt = -a / tau_a : 1
    """
    G_in = NeuronGroup(N_IN, eqs_in, threshold="v > v_th_in",
                       reset="v=0; a+=beta", refractory=2 * ms, method="euler",
                       namespace=ns_in)
    G_in.namespace["I_timed"] = TimedArray(np.zeros((1, N_IN), dtype=float), dt=DT_SIM)

    ns_h = dict(tau_h=P.tau_h_ms * ms, tau_vth=P.tau_vth_ms * ms,
                tau_r=P.tau_r_ms * ms, vth_rest=P.vth_rest, vth_jump=P.vth_jump,
                sigma_noise=P.sigma_noise * second ** -0.5)
    eqs_h = """
    dv/dt       = -v / tau_h + sigma_noise * xi   : 1
    dvth/dt     = -(vth - vth_rest) / tau_vth     : 1
    dtrace_r/dt = -trace_r / tau_r                : 1
    """
    G_h = NeuronGroup(N_H, eqs_h, threshold="v > vth",
                      reset="v=0; vth=vth+vth_jump; trace_r=1", method="euler",
                      namespace=ns_h)

    ns_s = dict(taupre=P.taupre_ms * ms, taupost=P.taupost_ms * ms,
                tau_x=P.tau_x_ms * ms, tau_y=P.tau_y_ms * ms, wmin=WMIN)
    stdp_model = """
    w          : 1
    dapre/dt   = -apre  / taupre  : 1 (event-driven)
    dapost/dt  = -apost / taupost : 1 (event-driven)
    dr1/dt     = -r1 / taupre      : 1 (event-driven)
    dr2/dt     = -r2 / tau_x       : 1 (event-driven)
    do1/dt     = -o1 / taupost     : 1 (event-driven)
    do2/dt     = -o2 / tau_y       : 1 (event-driven)
    wmax_syn   : 1
    Apre_syn   : 1
    Apost_syn  : 1
    A3pre_syn  : 1
    A3post_syn : 1
    """
    on_pre = ("v_post += w * (1 - trace_r_post)\n"
              "apre += Apre_syn\n"
              "w = clip(w + apost*(w-wmin) + A3post_syn*o1*r2*(w-wmin), wmin, wmax_syn)\n"
              "r1 += 1\nr2 += 1")
    on_post = ("apost += Apost_syn\n"
               "w = clip(w + apre*(wmax_syn-w) + A3pre_syn*r1*o2*(wmax_syn-w), wmin, wmax_syn)\n"
               "o1 += 1\no2 += 1")

    S_ih = Synapses(G_in, G_h, model=stdp_model, on_pre=on_pre, on_post=on_post,
                    namespace=ns_s)
    S_ih.connect(i=P.src_ih, j=P.tgt_ih)
    s_i, t_i = np.array(S_ih.i), np.array(S_ih.j)
    S_ih.wmax_syn   = P.wmax_m[s_i, t_i]
    S_ih.Apre_syn   = P.apre_m[s_i, t_i]
    S_ih.Apost_syn  = P.apost_m[s_i, t_i]
    S_ih.A3pre_syn  = P.a3pre_m[s_i, t_i]
    S_ih.A3post_syn = P.a3post_m[s_i, t_i]

    ns_i = dict(taupre=P.taupre_ms * ms, taupost=P.taupost_ms * ms, w_inh_min=W_INH_MIN)
    stdp_inh = """
    w_inh          : 1
    dapre_inh/dt   = -apre_inh  / taupre  : 1 (event-driven)
    dapost_inh/dt  = -apost_inh / taupost : 1 (event-driven)
    wmax_inh_syn   : 1
    Apre_inh_syn   : 1
    Apost_inh_syn  : 1
    """
    on_pre_inh = ("v_post -= w_inh * (1 - trace_r_post)\n"
                  "apre_inh += Apre_inh_syn\n"
                  "w_inh = clip(w_inh + apost_inh*(w_inh-w_inh_min), w_inh_min, wmax_inh_syn)")
    on_post_inh = ("apost_inh += Apost_inh_syn\n"
                   "w_inh = clip(w_inh + apre_inh*(wmax_inh_syn-w_inh), w_inh_min, wmax_inh_syn)")

    S_hh = Synapses(G_h, G_h, model=stdp_inh, on_pre=on_pre_inh, on_post=on_post_inh,
                    namespace=ns_i)
    S_hh.connect(i=P.src_hh, j=P.tgt_hh)
    s_h, t_h = np.array(S_hh.i), np.array(S_hh.j)
    S_hh.wmax_inh_syn  = P.wmax_inh[s_h, t_h]
    S_hh.Apre_inh_syn  = P.apre_inh_m[s_h, t_h]
    S_hh.Apost_inh_syn = P.apost_inh_m[s_h, t_h]
    S_hh.w_inh         = P.w_hh_init[s_h, t_h]

    spike_in, spike_hid = SpikeMonitor(G_in), SpikeMonitor(G_h)
    wmax_a, wmax_ia = np.array(S_ih.wmax_syn), np.array(S_hh.wmax_inh_syn)
    lim_exc, lim_inh = P.norm_limit_exc, P.norm_limit_inh

    @network_operation(dt=P.norm_dt_ms * ms, when='end')
    def normalize_weights():
        w = np.array(S_ih.w)
        cs = np.bincount(t_i, weights=w, minlength=N_H)
        sc = np.where(cs > lim_exc, lim_exc / cs, 1.0)
        S_ih.w[:] = np.clip(w * sc[t_i], WMIN, wmax_a)
        wi = np.array(S_hh.w_inh)
        csi = np.bincount(t_h, weights=wi, minlength=N_H)
        sci = np.where(csi > lim_inh, lim_inh / csi, 1.0)
        S_hh.w_inh[:] = np.clip(wi * sci[t_h], W_INH_MIN, wmax_ia)

    net = Network(G_in, G_h, S_ih, S_hh, spike_in, spike_hid, normalize_weights)
    G_h.vth = P.vth_rest
    net.store('init')
    return SimpleNamespace(net=net, P=P, G_in=G_in, G_h=G_h, S_ih=S_ih, S_hh=S_hh,
                           src_ih=s_i, tgt_ih=t_i, src_hh=s_h, tgt_hh=t_h,
                           spike_in=spike_in, spike_hid=spike_hid)


def warmup(h):
    h.S_ih.w     = h.P.w_ih_init[h.src_ih, h.tgt_ih]
    h.S_hh.w_inh = h.P.w_hh_init[h.src_hh, h.tgt_hh]
    h.G_in.namespace["I_timed"] = TimedArray(np.zeros((5, N_IN), dtype=float), dt=DT_SIM)
    h.net.run(5 * ms)
    h.net.restore('init')


def train_fingerprint(h, wav_path):
    P = h.P
    try:
        I, T = compute_spike_input_current(wav_path, **encoder_kwargs(P))
    except Exception as e:
        print(f"  [skip {wav_path}: {e}]")
        return None
    if T > P.clip_ms:
        T = P.clip_ms
        I = I[:, :T]

    h.net.restore('init')
    h.G_in.namespace["I_timed"] = TimedArray(
        np.tile(I, (1, P.n_epochs)).T.astype(float), dt=DT_SIM)
    h.S_ih.w     = P.w_ih_init[h.src_ih, h.tgt_ih]
    h.S_ih.apre  = 0; h.S_ih.apost = 0
    h.S_ih.r1 = 0; h.S_ih.r2 = 0; h.S_ih.o1 = 0; h.S_ih.o2 = 0
    h.S_hh.w_inh = P.w_hh_init[h.src_hh, h.tgt_hh]
    h.S_hh.apre_inh = 0; h.S_hh.apost_inh = 0

    h.net.run(P.n_epochs * T * DT_SIM)

    w_ih = np.zeros((N_IN, N_H), dtype=np.float32)
    w_ih[h.src_ih, h.tgt_ih] = np.array(h.S_ih.w)
    w_hh = np.zeros((N_H, N_H), dtype=np.float32)
    w_hh[h.src_hh, h.tgt_hh] = np.array(h.S_hh.w_inh)

    t0 = (P.n_epochs - 1) * T
    it, ii = np.array(h.spike_in.t / ms), np.array(h.spike_in.i)
    ht, hi = np.array(h.spike_hid.t / ms), np.array(h.spike_hid.i)
    ic = np.bincount(ii[it >= t0], minlength=N_IN).astype(np.int64)
    hc = np.bincount(hi[ht >= t0], minlength=N_H).astype(np.int64)
    return w_ih, w_hh, ic, hc


def _norm_act(c):
    d = np.percentile(c, 99)
    if d <= 0:
        return np.zeros_like(c, dtype=np.float16)
    return np.clip(c / d, 0.0, 1.0).astype(np.float16)


def fingerprint_to_sample(P, w_ih, w_hh, ic, hc):
    isil, hsil = ic == 0, hc == 0
    iw = w_ih[P.idx, P.j_row].astype(np.float32)
    iw[isil[P.idx] | hsil[None, None, :]] = 0.0
    hw = w_hh[P.idx, P.j_row].astype(np.float32)
    hw[hsil[P.idx] | hsil[None, None, :]] = 0.0
    return iw.astype(np.float16), hw.astype(np.float16), _norm_act(ic), _norm_act(hc)


# ── Worker plumbing ───────────────────────────────────────────────────────────
_H = None


def _init_worker(cfg):
    global _H
    _H = build_network(make_params(**cfg))
    warmup(_H)


def process_one(entry):
    pid, sid, fn, lab = entry['person_id'], entry['session_id'], entry['file_name'], entry['label']
    out = train_fingerprint(_H, entry['wav_path'])
    if out is None:
        return (pid, sid, fn, lab, None)
    return (pid, sid, fn, lab, fingerprint_to_sample(_H.P, *out))


In [ ]:
# ── Enumerate wavs: N speakers x SESSIONS each x UTTS per session ─────────────
# Unlike the main pipeline (1 random utterance per session), this KEEPS several
# utterances per session on purpose — that is what allows the within-session vs
# between-session comparison in the evaluation cell.
from pathlib import Path
import numpy as np

rng = np.random.default_rng(SEED)
root = Path(INPUT_ROOT)
assert root.is_dir(), f"INPUT_ROOT not found: {root}"

bases = sorted(d for d in root.glob("dev_*") if d.is_dir()) if DEV_NN_STYLE else [root]
assert bases, f"no dev_* folders under {root} — set DEV_NN_STYLE=False for a flat shard"

people = []
for b in bases:
    people += [d for d in sorted(b.iterdir()) if d.is_dir()]
assert len(people) >= N_SPEAKERS, f"only {len(people)} speakers under {root}"

entries = []
for pdir in rng.permutation(np.array(people, dtype=object)):
    sessions = [d for d in sorted(pdir.iterdir()) if d.is_dir()]
    usable = []
    for sdir in sessions:
        wavs = sorted(f.name for f in sdir.iterdir()
                      if f.is_file() and f.suffix.lower() in AUDIO_EXTS)
        if len(wavs) >= UTTS_PER_SESSION:
            usable.append((sdir, wavs))
    if len(usable) < SESSIONS_PER_SPK:
        continue                       # speaker cannot fill the quota — skip entirely
    for sdir, wavs in usable[:SESSIONS_PER_SPK]:
        pick = rng.choice(len(wavs), UTTS_PER_SESSION, replace=False)
        for w in (wavs[k] for k in pick):
            entries.append(dict(person_id=pdir.name, session_id=sdir.name, file_name=w,
                                label=f"{pdir.name}/{sdir.name}/{w}",
                                wav_path=str(sdir / w)))
    if len({e["person_id"] for e in entries}) >= N_SPEAKERS:
        break

n_spk = len({e["person_id"] for e in entries})
n_ses = len({(e["person_id"], e["session_id"]) for e in entries})
print(f"{len(entries)} wavs | {n_spk} speakers | {n_ses} sessions "
      f"| {len(entries)/max(n_ses,1):.1f} utts per session")
assert n_spk >= N_SPEAKERS, (
    f"only {n_spk} speakers had {SESSIONS_PER_SPK} sessions with >= "
    f"{UTTS_PER_SESSION} utterances each — lower the quotas")


In [ ]:
# ── How much audio does each clip_ms actually reach? ─────────────────────────
# clip_ms truncates but cannot invent audio. VoxCeleb1 utterances average ~8s, so
# 8000/16000 will not bind on every clip and the longer variants become variable-
# length. ffprobe is ~20ms per file, so this costs seconds and removes the guesswork
# that `s/wav` only answered indirectly in round 2.
import subprocess, concurrent.futures as cf
import numpy as np

def _dur_ms(path):
    try:
        out = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            capture_output=True, text=True, check=True).stdout.strip()
        return float(out) * 1000.0
    except Exception:
        return float("nan")

with cf.ThreadPoolExecutor(max_workers=16) as ex:
    _durs = np.array(list(ex.map(_dur_ms, [e["wav_path"] for e in entries])))
_d = _durs[np.isfinite(_durs)]
print(f"clip duration over {len(_d)} wavs (ms): "
      f"min {_d.min():.0f} | p25 {np.percentile(_d,25):.0f} | median {np.median(_d):.0f} "
      f"| p75 {np.percentile(_d,75):.0f} | max {_d.max():.0f}")
print(f"{'clip_ms':>8} {'% clips at full length':>23} {'mean audio actually used':>26}")
for _c in sorted({v.get("clip_ms", 2000) for v in VARIANTS}):
    _used = np.minimum(_d, _c)
    print(f"{_c:>8} {100*(_d >= _c).mean():>22.1f}% {_used.mean():>22.0f} ms")
print("\nA clip_ms whose 'mean audio actually used' is well below it is not really "
      "testing\nthat length — it is testing the corpus's own duration distribution.")


In [ ]:
# ── Run every variant over the SAME clips ────────────────────────────────────
# One npz per variant in OUT_DIR. Idempotent: an existing npz is reused, so a
# crashed or interrupted sweep resumes instead of restarting.
import multiprocessing as mp
import time
import numpy as np
import _fp_sweep_core as C

ctx = mp.get_context("spawn")

for cfg in VARIANTS:
    out_path = os.path.join(OUT_DIR, f"fp_{cfg['name']}.npz")
    if os.path.exists(out_path):
        # A seeded npz is only usable if it was built on THESE clips — otherwise the
        # "paired" comparison silently is not one. Skipped wavs make a cache a strict
        # subset, which is tolerable; anything else is rejected and recomputed.
        cached = set(np.load(out_path, allow_pickle=True)["labels"].tolist())
        want = set(e["label"] for e in entries)
        if not cached <= want:
            print(f"[{cfg['name']}] cache REJECTED — built on a different clip set "
                  f"({len(cached - want)} unknown labels); recomputing")
            os.remove(out_path)
        else:
            miss = len(want) - len(cached)
            print(f"[{cfg['name']}] cache hit -> {out_path}"
                  + (f"  [!] {miss} clips missing vs this run — rows not fully paired"
                     if miss else ""))
            continue

    print(f"\n[{cfg['name']}] n_epochs={cfg['n_epochs']} r_exc={cfg['r_exc']} "
          f"vth_rest={cfg['vth_rest']}", flush=True)
    t0 = time.time()
    # Warm the Cython cache in the parent so the workers all reuse it. Equations
    # embed vth_rest as a literal, so each distinct vth_rest compiles once.
    C.warmup(C.build_network(C.make_params(**cfg)))
    print(f"  warm-up {time.time()-t0:.0f}s", flush=True)

    rows, n_skip = [], 0
    t0 = time.time()
    with ctx.Pool(WORKERS, initializer=C._init_worker, initargs=(cfg,)) as pool:
        for k, (pid, sid, fn, lab, payload) in enumerate(
                pool.imap_unordered(C.process_one, entries), 1):
            if payload is None:
                n_skip += 1
            else:
                rows.append((*payload, pid, sid, fn, lab))
            if k % 50 == 0:
                el = time.time() - t0
                print(f"  {k}/{len(entries)}  {el:.0f}s  eta "
                      f"{el/k*(len(entries)-k):.0f}s", flush=True)

    assert rows, f"[{cfg['name']}] produced no fingerprints"
    np.savez(out_path,
             in_weights=np.stack([r[0] for r in rows]).astype(np.float16),
             hid_weights=np.stack([r[1] for r in rows]).astype(np.float16),
             input_activity=np.stack([r[2] for r in rows]).astype(np.float16),
             hidden_activity=np.stack([r[3] for r in rows]).astype(np.float16),
             person_ids=np.array([r[4] for r in rows]),
             session_ids=np.array([r[5] for r in rows]),
             file_names=np.array([r[6] for r in rows]),
             labels=np.array([r[7] for r in rows]),
             sec_per_wav=(time.time() - t0) / max(len(rows), 1),
             **{k: v for k, v in cfg.items() if k != "name"})
    print(f"[{cfg['name']}] {len(rows)} fingerprints ({n_skip} skipped) in "
          f"{time.time()-t0:.0f}s -> {out_path}")


In [ ]:
# ── Evaluate every variant on identical clips ────────────────────────────────
# All metrics are numpy-only. Every variant sees the same wavs, so the comparison
# is paired and the (small) speaker sample affects all rows equally.
import numpy as np

COMPONENTS = ("in_weights", "hid_weights", "input_activity", "hidden_activity")
# Block subsets worth scoring, keyed later by first letters: i=in_weights,
# h=hid_weights, i(nput_activity) and h(idden_activity) follow the same order.
SUBSETS = [
    ("in_weights",),
    ("in_weights", "hid_weights"),
    ("in_weights", "input_activity"),
    ("in_weights", "hid_weights", "input_activity"),
    ("in_weights", "hid_weights", "input_activity", "hidden_activity"),
]


def _flat(d, keys):
    n = len(d["person_ids"])
    parts = []
    for k in keys:
        a = np.nan_to_num(np.asarray(d[k], dtype=np.float32).reshape(n, -1))
        sd = a.std(0); mu = a.mean(0)
        parts.append((a - mu) / np.maximum(sd, 1e-6))    # z-score per block, then concat
    return np.concatenate(parts, axis=1)


def _pc_frac(X, ks=(5, 20)):
    Xc = X - X.mean(0)
    tot = (Xc ** 2).sum()
    if tot <= 0:
        return [float("nan")] * len(ks)
    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    ev = np.cumsum(s ** 2) / tot
    return [float(ev[min(k, len(ev)) - 1]) for k in ks]


def _fisher(X, spk):
    cls, inv = np.unique(spk, return_inverse=True)
    S, N, D = len(cls), *X.shape
    mu = X.mean(0)
    cnt = np.bincount(inv, minlength=S).astype(float)
    sums = np.zeros((S, D)); np.add.at(sums, inv, X)
    mu_s = sums / cnt[:, None]
    within = ((X - mu_s[inv]) ** 2).sum(0) / max(N - S, 1)
    between = (cnt[:, None] * (mu_s - mu) ** 2).sum(0) / max(S - 1, 1)
    live = X.std(0) > 1e-8
    return float(np.median((between / np.maximum(within, 1e-12))[live])) if live.any() else np.nan


def _retrieval(X, spk, rec):
    """Session-free all-pairs cosine retrieval — same protocol as the main notebook.
    Exact EER by sorting every valid pair."""
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    S = Xn @ Xn.T
    _, si = np.unique(rec, return_inverse=True)
    _, li = np.unique(spk, return_inverse=True)
    excl = si[:, None] == si[None, :]
    same = (li[:, None] == li[None, :]) & (~excl)
    Sm = np.where(excl, -np.inf, S)

    order = np.argsort(-Sm, axis=1)
    ss = np.take_along_axis(same, order, axis=1)
    rel = same.sum(1); ok = rel > 0
    ranks = np.arange(1, S.shape[0] + 1)
    prec = np.cumsum(ss, axis=1) / ranks
    ap = (prec * ss).sum(1) / np.maximum(rel, 1)

    sc = S[~excl]; y = same[~excl]
    o = np.argsort(-sc); ys = y[o].astype(float)
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    frr = 1 - tp / max(tp[-1], 1); far = fp / max(fp[-1], 1)
    k = int(np.argmin(np.abs(frr - far)))
    return dict(eer=float((frr[k] + far[k]) / 2),
                rank1=float(ss[ok, 0].mean()),
                rank5=float(ss[ok, :5].any(1).mean()),
                mAP=float(ap[ok].mean()))


def _session_convergence(X, spk, rec):
    """Mean cosine similarity for the three pair types. The design claims
    same-session pairs should be much more similar than different-session ones."""
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    S = Xn @ Xn.T
    _, si = np.unique(rec, return_inverse=True)
    _, li = np.unique(spk, return_inverse=True)
    off = ~np.eye(len(S), dtype=bool)
    same_sess = (si[:, None] == si[None, :]) & off
    same_spk_diff_sess = (li[:, None] == li[None, :]) & (si[:, None] != si[None, :])
    diff_spk = li[:, None] != li[None, :]
    f = lambda m: float(S[m].mean()) if m.any() else float("nan")
    return f(same_sess), f(same_spk_diff_sess), f(diff_spk)


def _session_average(X, spk, rec):
    """Average the fingerprints within each session -> one vector per session.

    Round 1's variance split (speaker 0.11 / session 0.21 / utterance noise 0.68)
    predicts that averaging N utterances scales the noise term by 1/N, so the
    speaker share of an N=2 average should rise 0.106 -> ~0.16. This measures it.
    Caveat when reading: the real application has ONE utterance at test time, so a
    win here is diagnostic, not directly shippable — `_nuisance_proj` is the
    per-utterance version of the same idea.
    """
    keys, inv = np.unique(rec, return_inverse=True)
    sums = np.zeros((len(keys), X.shape[1]), dtype=np.float64)
    np.add.at(sums, inv, X.astype(np.float64))
    Xs = (sums / np.bincount(inv, minlength=len(keys))[:, None]).astype(np.float32)
    spk_s = np.empty(len(keys), dtype=object)
    spk_s[inv] = spk                       # every row of a session shares its speaker
    return Xs, np.asarray(spk_s, dtype=str), keys


def _nuisance_proj(X, spk, rec, ks=(2, 5, 20), seed=0):
    """Learn the dominant WITHIN-speaker (nuisance) directions on half the speakers,
    project them out, and score the speaker-disjoint held-out half.

    Uses no session labels at scoring time, so unlike session averaging this
    transfers to the real application. Returns (EER before, EER after) on the same
    held-out speakers, so the two are directly comparable to each other — but NOT
    to the all-speaker EER in the main table.
    """
    cls = np.unique(spk)
    tr = set(np.random.default_rng(seed).permutation(cls)[: len(cls) // 2])
    m = np.array([s in tr for s in spk])
    Xtr = X[m]
    c_, inv = np.unique(spk[m], return_inverse=True)
    mu = np.zeros((len(c_), X.shape[1]), dtype=np.float64)
    np.add.at(mu, inv, Xtr.astype(np.float64))
    mu /= np.bincount(inv, minlength=len(c_))[:, None]
    W = Xtr - mu[inv].astype(np.float32)
    Vt = np.linalg.svd(W, full_matrices=False)[2]
    Xte, ste, rte = X[~m], spk[~m], rec[~m]
    out = {}
    for k in ks:
        V = Vt[: min(k, W.shape[0])]
        out[k] = _retrieval(Xte - (Xte @ V.T) @ V, ste, rte)["eer"]
    return _retrieval(Xte, ste, rte)["eer"], out



rows = []
for cfg in VARIANTS:
    p = os.path.join(OUT_DIR, f"fp_{cfg['name']}.npz")
    if not os.path.exists(p):
        print(f"[skip] {cfg['name']}: {p} missing"); continue
    d = np.load(p, allow_pickle=True)
    spk, rec = d["person_ids"], np.char.add(np.char.add(
        d["person_ids"], "/"), d["session_ids"])
    ha = np.asarray(d["hidden_activity"], dtype=np.float32)

    X = _flat(d, COMPONENTS)
    pc5, pc20 = _pc_frac(X)
    hpc5, _ = _pc_frac(np.asarray(d["hidden_activity"], dtype=np.float32))
    ipc5, _ = _pc_frac(np.asarray(d["input_activity"], dtype=np.float32))
    m = _retrieval(X, spk, rec)
    m_iw = _retrieval(_flat(d, ("in_weights",)), spk, rec)
    m_ia = _retrieval(_flat(d, ("input_activity",)), spk, rec)
    ss, sd_, dd = _session_convergence(X, spk, rec)

    # ── free post-processing: no new simulation, re-reads the cached npz ──────
    Xs, spk_s, rec_s = _session_average(X, spk, rec)
    eer_avg = _retrieval(Xs, spk_s, rec_s)["eer"]
    eer_ho, eer_proj = _nuisance_proj(X, spk, rec)
    comp = {c_: _retrieval(_flat(d, (c_,)), spk, rec)["eer"] for c_ in COMPONENTS}
    # Round 2 showed the 4-block concat is WORSE than in_weights alone on the 4s
    # variants, so search the subsets. Free — no simulation, just re-slicing.
    subsets = {"+".join(s[0] for s in sub): _retrieval(_flat(d, sub), spk, rec)["eer"]
               for sub in SUBSETS}

    rows.append(dict(
        name=cfg["name"], n=len(spk), dim=X.shape[1],
        eer=m["eer"], r1=m["rank1"], mAP=m["mAP"],
        eer_iw=m_iw["eer"], eer_ia=m_ia["eer"],
        pc5=pc5, pc20=pc20, ipc5=ipc5, hpc5=hpc5,
        dead=float((ha == 0).mean()), dead_all=float((ha.max(0) == 0).mean()),
        fisher=_fisher(X, spk), ss=ss, sd=sd_, dd=dd,
        sec=float(d["sec_per_wav"]),
        eer_avg=eer_avg, eer_ho=eer_ho, eer_proj=eer_proj, comp=comp,
        subsets=subsets,
    ))

print("MAIN COMPARISON  (all variants on identical clips — paired)\n")
print(f"{'variant':>12} {'EER':>7} {'R@1':>7} {'mAP':>7} | {'EER iw':>7} {'EER ia':>7} | "
      f"{'PC1-5':>6} {'PC1-20':>7} {'hid PC5':>8} | {'dead':>6} {'F':>5} | {'s/wav':>6}")
print("-" * 108)
for r in rows:
    print(f"{r['name']:>12} {r['eer']:>7.4f} {r['r1']:>7.4f} {r['mAP']:>7.4f} | "
          f"{r['eer_iw']:>7.4f} {r['eer_ia']:>7.4f} | {r['pc5']:>6.3f} {r['pc20']:>7.3f} "
          f"{r['hpc5']:>8.3f} | {r['dead']*100:>5.1f}% {r['fisher']:>5.2f} | {r['sec']:>6.1f}")
print("-" * 108)
print("EER/R@1/mAP  : cross-session speaker retrieval on the fingerprint alone. LOWER EER better.")
print("EER iw / ia  : same, using only in_weights / only input_activity.")
print("PC1-5/PC1-20 : share of variance in the top components = redundancy. LOWER IS BETTER —")
print("               this is the quantity change #1 (narrower receptive field) targets.")
print("hid PC5      : same for hidden_activity alone; baseline had 0.856 vs 0.684 for its own")
print("               input, i.e. the hidden layer was MORE redundant than what fed it.")
print("dead         : fraction of (clip, hidden unit) pairs with zero spikes. LOWER better.")
print("F            : between-speaker / within-speaker variance. 1.0 = no speaker structure.")

print("\n\nSESSION CONVERGENCE  (the design's own claim, measurable here for the first time)\n")
print(f"{'variant':>12} {'same session':>13} {'same spk, diff sess':>21} {'diff speaker':>13} "
      f"{'session gap':>12}")
print("-" * 78)
for r in rows:
    print(f"{r['name']:>12} {r['ss']:>13.4f} {r['sd']:>21.4f} {r['dd']:>13.4f} "
          f"{r['ss']-r['sd']:>12.4f}")
print("-" * 78)
print("Mean cosine similarity between fingerprints, by pair type.")
print("The claim is that utterances from the SAME session converge to nearly the same")
print("fingerprint. That holds if `same session` is clearly above `same spk, diff sess`.")
print("A large session gap also means session-level AVERAGING is available as free denoising.")

print("\n\nDENOISING & PER-COMPONENT  (post-processing only — no extra simulation)\n")
print(f"{'variant':>12} {'EER':>7} {'sess-avg':>9} | {'held-out':>9} "
      + " ".join(f"{'proj-'+str(k):>8}" for k in (2, 5, 20))
      + " | " + " ".join(f"{c_[:9]:>9}" for c_ in COMPONENTS))
print("-" * 122)
for r in rows:
    print(f"{r['name']:>12} {r['eer']:>7.4f} {r['eer_avg']:>9.4f} | {r['eer_ho']:>9.4f} "
          + " ".join(f"{r['eer_proj'][k]:>8.4f}" for k in (2, 5, 20))
          + " | " + " ".join(f"{r['comp'][c_]:>9.4f}" for c_ in COMPONENTS))
print("-" * 122)
print("sess-avg : EER after averaging the 2 utterances of each session into one vector.")
print("           Diagnostic only — the real application has one utterance at test time.")
print("held-out / proj-k : EER on a speaker-disjoint half, before and after projecting out")
print("           the top k within-speaker directions learned on the other half. Compare")
print("           these to EACH OTHER, not to the EER column. proj-20 HURT every variant")
print("           in round 2; k=2 and k=5 test whether it was simply too aggressive.")
print("last 4   : EER from each fingerprint block alone. A block near 0.5 is pure noise.")

print("\n\nBLOCK SUBSETS  (which parts of the fingerprint are worth keeping)\n")
_keys = list(rows[0]["subsets"]) if rows else []
_lbl = {1: "in_w", 2: "in+hid_w", 3: "in_w+in_act", 4: "in+hid_w+in_act", 5: "all four"}
print(f"{'variant':>12} " + " ".join(f"{_lbl[i+1]:>16}" for i in range(len(_keys))))
print("-" * (13 + 17 * len(_keys)))
for r in rows:
    best = min(r["subsets"], key=r["subsets"].get)
    print(f"{r['name']:>12} " + " ".join(
        f"{r['subsets'][k]:>16.4f}" + ("*" if k == best else " ") for k in _keys))
print("-" * (13 + 17 * len(_keys)))
print("* = best subset for that variant. Round 2 found in_weights ALONE beat all four")
print("  blocks concatenated on both 4s variants (0.3458 vs 0.3601), i.e. the weaker")
print("  blocks dilute rather than add. If that repeats here, drop them from the tensor.")

# ── Simulation reproducibility ────────────────────────────────────────────────
# Rounds 1-3 measured the noise floor only on EER, an aggregate. This measures it on
# the fingerprints themselves: two runs of an IDENTICAL config differ only by the
# unseeded sigma_noise. Cosine between the same wav's two fingerprints says how much
# of the representation is reproducible signal versus simulation noise — which is
# what decides whether `sigma_noise` is worth tuning or is a red herring.
if REPLICATES:
    print("\n\nSIMULATION REPRODUCIBILITY  (identical config, different RNG)\n")
    for _a, _b in REPLICATES:
        _pa = os.path.join(OUT_DIR, f"fp_{_a}.npz")
        _pb = os.path.join(OUT_DIR, f"fp_{_b}.npz")
        if not (os.path.exists(_pa) and os.path.exists(_pb)):
            print(f"  {_a} / {_b}: missing npz, skipped"); continue
        _da, _db = np.load(_pa, allow_pickle=True), np.load(_pb, allow_pickle=True)
        _la, _lb = list(_da["labels"]), list(_db["labels"])
        _common = sorted(set(_la) & set(_lb))
        _ia = np.array([_la.index(x) for x in _common])
        _ib = np.array([_lb.index(x) for x in _common])
        _Xa, _Xb = _flat(_da, COMPONENTS)[_ia], _flat(_db, COMPONENTS)[_ib]
        _u = lambda M: M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-12)
        _na, _nb = _u(_Xa), _u(_Xb)
        _same = float((_na * _nb).sum(1).mean())
        _S = _na @ _na.T
        _off = ~np.eye(len(_S), dtype=bool)
        print(f"  {_a} vs {_b}   ({len(_common)} wavs)")
        print(f"    same wav, two runs      cos {_same:>7.4f}   <- reproducible part")
        print(f"    different wavs, one run cos {_S[_off].mean():>7.4f}   <- shared baseline")
        print(f"    => {100*(1-_same):.1f}% of the fingerprint does not survive a re-run.")
    print("\n  If that share is small, the ~59% within-utterance variance measured in")
    print("  rounds 2-3 is real acoustic content, not simulation noise, and lowering")
    print("  sigma_noise cannot help. If it is large, `noise_lo` should win outright.")


## How to read the result

### Read the noise floor first

`champ` and `champ_rep` are the same config. Their gap is this round's resolution limit; it has
come back at **0.0031-0.0035** four rounds running. **A variant counts as real only if it beats
the pair mean by more than 2x that (~0.007).** The champion's own number should land near round
5's **0.3330** — if it does not, something about the sample or the corpus mount changed and the
round is not comparable to the diary.

The same pair also gives an **R@1 floor** (0.0062 last round). Round 5's `norm_exc_15` was an EER
tie with a 4.2x-floor R@1 gain, so check both columns: EER is a threshold statistic over all
pairs, R@1 says whether the right speaker actually reaches rank 1.

### The round's real test

**Does any family outside "activity" respond at all?** Five rounds have closed the encoder, the
activity axis, exposure length, `sigma_noise`, and normalisation timing. These four are new:

- **`beta_lo` / `tau_a_120`** — the input layer's adaptive LIF has never been touched. `beta` is
  how hard a neuron adapts after firing; `tau_a` is how long it stays adapted. Together they set
  how much the front end reports *change* versus *steady state*. Watch `EER ia` (input_activity
  alone), which is the block these two act on most directly and which has been stuck at
  0.380 ± 0.002 for the entire loop.
- **`vthj_2` / `tau_vth_200`** — hidden spike-frequency adaptation. Read these differently from
  round 5's knobs: they reduce activity *selectively*, silencing whichever units are currently
  most active, so they are a competition mechanism rather than a uniform drive change. The
  inverted-U finding does not automatically predict them. If `dead` rises but EER improves, that
  is the signature of real competition and it contradicts the round-4 line in an informative way.
- **`triplet_off`** — a tie here means the learning rule can be simplified with no cost, which is
  worth knowing even though it is not an improvement.
- **`p_exc_1`** — the only remaining handle on defect 1. It sharpens the topographic profile
  without cutting fan-in, so unlike `r_exc` it should not kill units. Watch **`hid PC5`**: this
  is the variant that is *supposed* to lower it. If EER moves without `hid PC5` moving, distrust
  the mechanism story even if you keep the number.

### The combination row

`vth_nexc` stacks round 5's two sub-threshold trends. If it lands beyond 2x floor, sub-threshold
agreement across EER/R@1/mAP is a usable signal and future rounds can combine on it. If it lands
at a tie like `inh_weak` and `stdp_2x` did, that heuristic is dead and one-factor-at-a-time is
the only thing to trust.

### The denoising table — now the most interesting one

`proj-20` flipped from harmful (round 2) to a consistent ≈5x-floor gain (round 5, 8 of 9
variants). Compare `held-out` to `proj-k` **for each variant**, never to the `EER` column. The
best number of the loop is currently `vth_035` held-out 0.3354 → proj-20 **0.3125**.

This is the one result so far that is both shippable and free: the projection is learned on
training speakers and needs no session labels at test time, unlike `sess-avg`.

### The block-subset table

`hid_weights` and `hidden_activity` have diluted on **every variant of every round**. Best
subset is `in_weights` alone or `in_weights + input_activity`, depending on variant; round 5's
best was `norm_exc_15` on `in_weights` alone at **0.3132**. The caveat that keeps this from being
a decision: `ecapa_film_snn` uses a *learned* readout, which may find structure unsupervised
cosine cannot.

### Stopping rule (`PROTOCOL.md`, revised after round 3)

The loop stops when **two consecutive rounds** produce no variant beating the champion by more
than 2x the noise floor. **Round 5 produced none, so the counter is at 1 of 2.** If this round is
also empty, the loop ends and the answer is that the fingerprint's remaining weakness is not in
this hyperparameter space.

The target is unchanged: regenerate one shard with the champion, run
`../ecapa_film_snn/ecapa_film_snn.ipynb` cell 14 Part B, and check whether fingerprint-alone EER
drops below **0.28** (from 0.345). That promotion test has still never been run, and with the
sweep at −17% relative and the proj-20 gain on top of it, it is now the highest-value measurement
available — it runs on a different machine path and does not need the sweep to finish.

### Caveat

60 speakers, by design. Fine for **ranking** variants that share identical clips; the absolute
EER values are not corpus-level results and should not be quoted as such.
